# CIPHER - Kaggle Benchmark Notebook
**Calibrated Introspection via Partially Hidden Environment Rules**

Dataset attached at `/kaggle/input/datasets/wenhaolu49/cipher`.

| Dimension | Weight | What it measures |
|-----------|--------|------------------|
| Objective | 35% | Plan quality vs. oracle beam search |
| Calibration | 25% | Brier score on stated self-knowledge |
| Attention | 20% | Rank correlation on unknown importance |
| Executive | 20% | Plan structure, named risks, alternative plans |

In [ ]:
# 1. Install dependencies
import subprocess, sys

def _pip(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True)

_pip("kaggle-benchmarks", "pandas", "matplotlib", "numpy")
print("Dependencies installed.")

In [ ]:
# 2. Imports
import os, sys, json
import pandas as pd
import kaggle_benchmarks as kbench
from typing import Any, Dict, List, Optional
from kaggle_benchmarks.prompting import ResponseParsingError

_KAGGLE_ROOT = "/kaggle/input/datasets/wenhaolu49/cipher"
_LOCAL_ROOT  = os.path.abspath(os.getcwd())
_LOCAL_PARENT= os.path.abspath(os.path.join(os.getcwd(), ".."))

CIPHER_ROOT = None
for _c in [_KAGGLE_ROOT, _LOCAL_ROOT, _LOCAL_PARENT]:
    if os.path.isdir(os.path.join(_c, "cipher")):
        CIPHER_ROOT = _c
        break
if CIPHER_ROOT is None:
    raise RuntimeError("cipher/ package not found - attach the cipher dataset.")
if CIPHER_ROOT not in sys.path:
    sys.path.insert(0, CIPHER_ROOT)
print(f"cipher root: {CIPHER_ROOT}")

from cipher.generator import Instance
from cipher.world import World, State, EntityState, Rule, Trigger, Effect
from cipher.schema import validate_response
from cipher.scorer import score_response
print("cipher imports OK.")

In [ ]:
# 3. Load instances
DATA_PATH = next(
    (p for p in [
        os.path.join(_KAGGLE_ROOT, "data", "instances.jsonl"),
        os.path.join(CIPHER_ROOT,  "data", "instances.jsonl"),
    ] if os.path.exists(p)),
    None,
)
if DATA_PATH is None:
    raise FileNotFoundError("data/instances.jsonl not found.")

KBENCH_LIMIT = 1000

with open(DATA_PATH) as f:
    ALL_RECORDS = [json.loads(line) for line in f if line.strip()]
if KBENCH_LIMIT:
    ALL_RECORDS = ALL_RECORDS[:KBENCH_LIMIT]

print(f"Loaded {len(ALL_RECORDS)} instances from {DATA_PATH}")
print("\nAvailable models:", list(kbench.llms.keys()))

In [ ]:
# 4. Scoring helpers
import re as _re

def _extract_outermost_json(text: str) -> str:
    """Find the first { and scan forward to its matching }, handling strings and escapes."""
    start = text.find("{")
    if start == -1:
        return ""
    depth, in_string, escape = 0, False, False
    for i, ch in enumerate(text[start:], start):
        if escape:
            escape = False
            continue
        if ch == "\\" and in_string:
            escape = True
            continue
        if ch == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    return text[start:]

def _parse_json_tolerant(text: str):
    """Strip // comments and trailing commas, fall back to single-quote replacement."""
    cleaned = _re.sub(r'//[^\n]*', '', text)
    cleaned = _re.sub(r',\s*([}\]])', r'\1', cleaned)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return json.loads(cleaned.replace("'", '"'))

def _instance_from_record(rec: Dict[str, Any]) -> Instance:
    h = rec["hidden"]
    rules = []
    for r in h["rules"]:
        t, e = r["trigger"], r["effect"]
        rules.append(Rule(
            name=r["name"],
            trigger=Trigger(kind=t["kind"], i=t["i"], j=t.get("j", -1), k=t.get("k", 0)),
            effect=Effect(kind=e["kind"], target=e["target"],
                          delta=e.get("delta", 0), source=e.get("source", -1)),
        ))
    initial = State(tuple(EntityState(phase=s["phase"], flux=s["flux"]) for s in h["initial_state"]))
    return Instance(
        id=rec["id"], seed=rec["seed"], difficulty=rec["difficulty"],
        world=World(initial=initial, rules=tuple(rules), horizon=h["horizon"]),
        visible_rule_indices=h["visible_rule_indices"],
        hidden_rule_indices=h["hidden_rule_indices"],
        public_rule_descriptions=[],
        hidden_fields=h.get("hidden_fields", []),
        metacog_ground_truth=h["metacog_ground_truth"],
        true_unknown_ranking=h["true_unknown_ranking"],
        oracle_objective=h.get("oracle_best"),
    )

_ZERO = dict(composite=0.0, objective=0.0, calibration=0.0, attention=0.0, executive=0.0)

def _score(raw_text: str, rec: Dict[str, Any]) -> Dict[str, float]:
    blob = _extract_outermost_json(raw_text)
    if not blob:
        return _ZERO.copy()
    try:
        raw  = _parse_json_tolerant(blob)
        inst = _instance_from_record(rec)
        resp = validate_response(raw)
        bd   = score_response(resp, inst,
                              best_obj=rec["hidden"].get("oracle_best"),
                              worst_obj=rec["hidden"].get("oracle_worst"))
        return bd.to_dict()
    except Exception:
        return _ZERO.copy()

In [ ]:
# 5. Per-instance task
@kbench.task(name="cipher_single_instance_scorer_v2", store_task=False)
def cipher_single_instance_scorer(llm, prompt: str, hidden_json: str) -> dict:
    try:
        reply = llm.prompt(prompt)
        rec   = json.loads(hidden_json)
        return _score(str(reply), rec)
    except (ResponseParsingError, json.JSONDecodeError, Exception):
        return _ZERO.copy()

print("cipher_single_instance_scorer defined.")

In [ ]:
# 6. Batch evaluation
# Set to an integer (e.g. 10) for a fast qualitative run; None = full 1000-instance eval
QUALITATIVE_N = 100

_LAST_RUN = {}

@kbench.task(name="cipher_eval_full")
def cipher_metacognition_evaluation(llm):
    """CIPHER: micro-worlds with hidden causal rules in invented vocabulary. Tests metacognitive calibration."""
    records_to_run = ALL_RECORDS[:QUALITATIVE_N] if QUALITATIVE_N else ALL_RECORDS
    evaluation_df = pd.DataFrame([{
        "prompt":      rec["prompt"],
        "hidden_json": json.dumps({k: rec[k] for k in ["id", "seed", "difficulty", "hidden"]}),
    } for rec in records_to_run])

    with kbench.client.enable_cache():
        runs = cipher_single_instance_scorer.evaluate(
            llm=[llm],
            evaluation_data=evaluation_df,
            n_jobs=8,
            remove_run_files=True,
        )

    results_df = runs.as_dataframe()

    per_instance = []
    for rec, result in zip(records_to_run, results_df["result"]):
        row = {"id": rec["id"], "difficulty": rec["difficulty"]}
        if isinstance(result, dict):
            row.update(result)
        else:
            row.update(_ZERO)
        per_instance.append(row)

    _LAST_RUN["per_instance"] = per_instance

    _out_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.join(os.getcwd(), "analysis")
    os.makedirs(_out_dir, exist_ok=True)
    _model_tag = getattr(llm, "model_id", getattr(llm, "name", "unknown")).replace("/", "-")
    _out_path = os.path.join(_out_dir, f"per_instance_{_model_tag}.json")
    with open(_out_path, "w") as _f:
        json.dump(per_instance, _f, indent=2)

    def _mean(key, subset=None):
        rows = [r for r in per_instance if subset is None or r["difficulty"] == subset]
        return float(sum(r.get(key, 0.0) for r in rows) / max(len(rows), 1))

    scores = {k: _mean(k) for k in ["composite", "objective", "calibration", "attention", "executive"]}
    _LAST_RUN["scores"] = scores
    _LAST_RUN["n"] = len(per_instance)
    _LAST_RUN["_mean"] = _mean

In [ ]:
%choose cipher_eval_full
cipher_metacognition_evaluation.run(kbench.llm)

scores = _LAST_RUN.get("scores", {})
n     = _LAST_RUN.get("n", 0)
_mean = _LAST_RUN.get("_mean")

print(f"\n=== CIPHER Results (n={n}) ===")
print(f"{'Dimension':<14} {'Score':>6}")
print("-" * 22)
for dim in ["composite", "objective", "calibration", "attention", "executive"]:
    print(f"{dim.capitalize():<14} {scores.get(dim, 0.0):>6.3f}")

if _mean:
    print("\n=== By Difficulty ===")
    print(f"{'Difficulty':<10} {'Composite':>9} {'Objective':>9} {'Calibration':>11} {'Attention':>9} {'Executive':>9}")
    print("-" * 60)
    for diff in ["easy", "medium", "hard"]:
        n_diff = sum(1 for r in _LAST_RUN["per_instance"] if r["difficulty"] == diff)
        if n_diff == 0:
            continue
        row = {k: _mean(k, diff) for k in ["composite", "objective", "calibration", "attention", "executive"]}
        print(f"{diff:<10} {row['composite']:>9.3f} {row['objective']:>9.3f} {row['calibration']:>11.3f} {row['attention']:>9.3f} {row['executive']:>9.3f}  (n={n_diff})")